# Project: Meeting Prep Briefer
**Brief from Paras:**
Runs every morning at 7am. Reads today's Calendar, looks up each attendee on web,
pulls past Gmail threads + Notion notes, sends one-page brief per meeting.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class MeetingPrepBrieferState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    meetings: list[dict]
    current_meeting: Optional[dict]
    attendee_research: Optional[str]
    history_summary: Optional[str]
    brief: Optional[str]


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# calendar_reader - Calendar reader
calendar_reader_tools = toolset.get_tools(actions=[Action.GOOGLECALENDAR_LIST_EVENTS])
calendar_reader_agent = create_react_agent(llm, calendar_reader_tools, prompt='''Fetch today's meetings, return list with attendees and title.''')
def calendar_reader_node(state):
    result = calendar_reader_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='calendar_reader')]}


# attendee_researcher - Web + Notion attendee researcher
attendee_researcher_tools = toolset.get_tools(actions=[Action.TAVILY_TAVILY_SEARCH, Action.NOTION_SEARCH_NOTION_PAGE])
attendee_researcher_agent = create_react_agent(llm, attendee_researcher_tools, prompt='''Search web + Notion for each attendee. Return 5-bullet brief on each.''')
def attendee_researcher_node(state):
    result = attendee_researcher_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='attendee_researcher')]}


# history_puller - Gmail thread puller
history_puller_tools = toolset.get_tools(actions=[Action.GMAIL_FETCH_EMAILS])
history_puller_agent = create_react_agent(llm, history_puller_tools, prompt='''Pull past 5 emails between user and attendees. Summarise.''')
def history_puller_node(state):
    result = history_puller_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='history_puller')]}


# composer - Brief composer + Slack sender
composer_tools = toolset.get_tools(actions=[Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL])
composer_agent = create_react_agent(llm, composer_tools, prompt='''Compose one-page brief per meeting. Send to user's Slack DM.''')
def composer_node(state):
    result = composer_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='composer')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if not state.get('meetings'): return {'next_worker': 'calendar_reader'}
    if state.get('attendee_research') is None: return {'next_worker': 'attendee_researcher'}
    if state.get('history_summary') is None: return {'next_worker': 'history_puller'}
    if state.get('brief') is None: return {'next_worker': 'composer'}
    return {'next_worker': 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'attendee_researcher', 'composer', 'history_puller', 'calendar_reader'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("calendar_reader", calendar_reader_node)
g.add_node("attendee_researcher", attendee_researcher_node)
g.add_node("history_puller", history_puller_node)
g.add_node("composer", composer_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "calendar_reader": "calendar_reader",
    "attendee_researcher": "attendee_researcher",
    "history_puller": "history_puller",
    "composer": "composer",
    "__end__": END,
})
g.add_edge("calendar_reader", "supervisor")
g.add_edge("attendee_researcher", "supervisor")
g.add_edge("history_puller", "supervisor")
g.add_edge("composer", "supervisor")

conn = sqlite3.connect("03_meeting_prep_briefer.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '03_meeting_prep_briefer-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'messages': [HumanMessage(content='Run morning brief for today')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
